In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
import zipfile

In [2]:
# ============================================================
# LOAD DATA
# ============================================================

In [3]:
# Download the dataset directly from UCI repository
url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
urllib.request.urlretrieve(url, "bank-marketing.zip")

('bank-marketing.zip', <http.client.HTTPMessage at 0x1a57aec7250>)

In [4]:
# Extract outer zip
with zipfile.ZipFile("bank-marketing.zip","r") as z:
    z.extractall("bank_marketing_data")

# Extract inner zip (bank-additional.zip contains the file we need)
with zipfile.ZipFile("bank_marketing_data/bank-additional.zip","r") as z:
    z.extractall("bank_marketing_data")

In [5]:
df = pd.read_csv("bank_marketing_data/bank-additional/bank-additional-full.csv", sep=";")
print(df.shape)
df.head()

(41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [6]:
# ============================================================
# TARGET DISTRIBUTION
# ============================================================

In [7]:
# Check class balance for the target variable
df["y"].value_counts(normalize=True)

y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64

In [8]:
# ============================================================
# MISSING VALUES CHECK
# ============================================================

In [9]:
# This dataset encodes missing values as the string "unknown" instead of NaN
# Check which categorical columns contain "unknown" and how frequent it is
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    unknown_count = (df[col] == "unknown").sum()
    if unknown_count > 0:
        print(f"{col}:{unknown_count} unknwon ({unknown_count/len(df)*100:.2f}%)")


job:330 unknwon (0.80%)
marital:80 unknwon (0.19%)
education:1731 unknwon (4.20%)
default:8597 unknwon (20.87%)
housing:990 unknwon (2.40%)
loan:990 unknwon (2.40%)


In [10]:
# ============================================================
# HANDLE MISSING VALUES ("unknown")
# ============================================================

In [11]:
# Check the value distribution of 'default' - is "yes" even a meaningful category?
print(df["default"].value_counts())

default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64


In [12]:
# Drop rows with "unknown" in job and marital (very low frequency, safe to drop)
df = df[(df["job"] != "unknown") & (df["marital"] != "unknown")]

In [13]:
print(df.shape)

(40787, 21)


In [14]:
# ============================================================
# SIMPLIFY 'default' COLUMN
# ============================================================

In [15]:
# "yes" is extremely rare (only 3 rows) and adds noise/instability to the model
# Merge "yes" into "unknown" to keep the column effectively binary (no vs unknown)
df["default"] = df["default"].replace("yes", "unknown")

In [16]:
print(df["default"].value_counts())

default
no         32348
unknown     8439
Name: count, dtype: int64


In [17]:
# ============================================================
# NUMERIC COLUMNS OVERVIEW
# ============================================================

In [18]:
# Get statistical summary of numeric columns to spot potential outliers and scale differences
numeric_cols = df.select_dtypes(include=["int64","float64"]).columns
print(numeric_cols.tolist())

['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


In [19]:
df[numeric_cols].describe()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,40787.000000,40787.000000,40787.000000,40787.000000,40787.000000,40787.000000,40787.000000,40787.000000,40787.000000,40787.000000
mean,39.978817,258.329811,2.566112,962.725427,0.172874,0.080516,93.574804,-40.515770,3.619532,5167.017866
std,10.402157,259.269596,2.768103,186.293432,0.494863,1.570133,0.578558,4.626805,1.734416,72.231843
min,17.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.000000,102.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.000000,180.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,319.500000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.000000,4918.000000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


In [20]:
# ============================================================
# DROP LEAKAGE COLUMN & HANDLE 'pdays' SENTINEL VALUE
# ============================================================

In [21]:
# 'duration' causes data leakage: call duration is only known AFTER the call ends,
# so it cannot be used for a realistic pre-call prediction model
df = df.drop(columns=["duration"])

In [22]:
# 'pdays' = 999 is a sentinel value meaning "client was not previously contacted"
# Treating it as a real numeric value (999 days) would distort the model
# Create a binary flag instead, then keep pdays but it will mostly separate via this flag
df["was_previously_contacted"] = (df["pdays"] != 999).astype(int)

In [23]:
print(df["was_previously_contacted"].value_counts())

was_previously_contacted
0    39297
1     1490
Name: count, dtype: int64


In [24]:
# ============================================================
# CHECK FOR COLLISION BEFORE REPLACING SENTINEL
# ============================================================

In [25]:
# Check if any actually-contacted client has pdays == 0 (real "contacted today" case)
# This would collide with our planned sentinel replacement
collision = df[(df["pdays"] == 0) & (df["was_previously_contacted"] == 0)]
print("Rows with pdays=0 among 'never contacted':", len(collision))

Rows with pdays=0 among 'never contacted': 0


In [26]:
real_zero = df[(df["pdays"] == 0) & (df["was_previously_contacted"] == 1)]
print("Rows with pdays=0 among 'actually contacted':", len(real_zero))

Rows with pdays=0 among 'actually contacted': 15


In [27]:
# ============================================================
# FIX COLLISION: USE -1 AS SENTINEL, KEEP REAL pdays=0 INTACT
# ============================================================

In [28]:
# The previous replace(999, 0) collided with genuine "contacted same day" cases (pdays=0)
# Fix: only rows where was_previously_contacted == 0 are true sentinels -> set them to -1
# This leaves real pdays=0 (contacted same day) untouched
df.loc[df["was_previously_contacted"] == 0, "pdays"] = -1

In [29]:
# Verify: no more collision, and real zero cases preserved
print(df["pdays"].value_counts().head())
print("Rows with pdays=0 among 'actually contacted':", 
      len(df[(df["pdays"] == 0) & (df["was_previously_contacted"] == 1)]))

pdays
-1    39297
 3      431
 6      404
 4      116
 9       64
Name: count, dtype: int64
Rows with pdays=0 among 'actually contacted': 15


In [30]:
# ============================================================
# OUTLIER CHECK (IQR METHOD) — age & campaign
# ============================================================

In [31]:
# Check outlier boundaries for 'age' and 'campaign' using IQR method
for col in ["age","campaign"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_count = ((df[col] < lower ) | ( df[col] > upper )).sum()
    print(f"{col}: lower={lower:.2f}, upper={upper:.2f}, outliers={outlier_count} ({outlier_count/len(df)*100:.2f}%)")

age: lower=9.50, upper=69.50, outliers=463 (1.14%)
campaign: lower=-2.00, upper=6.00, outliers=2372 (5.82%)


In [32]:
# ============================================================
# OUTLIER CAPPING (IQR METHOD) — age & campaign
# ============================================================

In [33]:
# Cap outliers instead of removing them, to avoid losing data
# 'age' outliers likely represent real retired clients, not data errors
# 'campaign' outliers represent extreme contact counts, capped to reduce distance-metric distortion (important for SVM)

In [34]:
for col in ["age","campaign"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower, upper=upper)

df[["age","campaign"]].describe()

,age,campaign
count,40787.000000,40787.000000
mean,39.894611,2.274818
std,10.110494,1.548989
min,17.000000,1.000000
25%,32.000000,1.000000
50%,38.000000,2.000000
75%,47.000000,3.000000
max,69.500000,6.000000


In [35]:
# ============================================================
# MULTICOLLINEARITY CHECK — CORRELATION MATRIX
# ============================================================

In [36]:
# Check correlation among numeric features, especially the macroeconomic indicators
# which likely move together since they reflect the same economic period
numeric_cols = ["age", "campaign", "pdays", "previous", "emp.var.rate",
                 "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]

In [37]:
corr_matrix = df[numeric_cols].corr()
corr_matrix

,age,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
age,1.000000,0.002173,0.014429,0.015834,0.011752,0.005781,0.125775,0.023601,-0.002410
campaign,0.002173,1.000000,-0.049598,-0.083570,0.147825,0.113959,-0.018507,0.128729,0.141819
pdays,0.014429,-0.049598,1.000000,0.505147,-0.228574,-0.047253,0.069662,-0.260781,-0.333639
previous,0.015834,-0.083570,0.505147,1.000000,-0.419388,-0.202222,-0.051712,-0.453471,-0.500273
emp.var.rate,0.011752,0.147825,-0.228574,-0.419388,1.000000,0.775333,0.197914,0.972232,0.906822
cons.price.idx,0.005781,0.113959,-0.047253,-0.202222,0.775333,1.000000,0.060993,0.688048,0.521586
cons.conf.idx,0.125775,-0.018507,0.069662,-0.051712,0.197914,0.060993,1.000000,0.279187,0.101975
euribor3m,0.023601,0.128729,-0.260781,-0.453471,0.972232,0.688048,0.279187,1.000000,0.945140
nr.employed,-0.002410,0.141819,-0.333639,-0.500273,0.906822,0.521586,0.101975,0.945140,1.000000


In [38]:
# ============================================================
# MULTICOLLINEARITY CHECK — VIF
# ============================================================

In [39]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [40]:
vif_data = pd.DataFrame()
vif_data["feature"] = numeric_cols
vif_data["VIF"] = [variance_inflation_factor(df[numeric_cols].values, i) for i in range(len(numeric_cols))]

vif_data.sort_values("VIF", ascending=False)

,feature,VIF
8,nr.employed,26677.268187
5,cons.price.idx,22665.609091
7,euribor3m,227.198858
6,cons.conf.idx,119.589040
4,emp.var.rate,29.076439
0,age,16.852784
1,campaign,3.258892
3,previous,1.805373
2,pdays,1.750394


In [41]:
# ============================================================
# DROP REDUNDANT MACROECONOMIC FEATURES
# ============================================================

In [42]:
# Keep only 'euribor3m' among the 5 macroeconomic indicators:
# it's the most interpretable (market interest rate) and most correlated with the others,
# so dropping the rest minimizes information loss while removing severe multicollinearity
df = df.drop(columns=["emp.var.rate", "cons.price.idx", "cons.conf.idx", "nr.employed"])

In [43]:
print(df.shape)

(40787, 17)


In [44]:
df.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'euribor3m',
 'y',
 'was_previously_contacted']

In [45]:
# ============================================================
# INSPECT CATEGORICAL COLUMNS BEFORE ENCODING
# ============================================================

In [46]:
# Check unique values and their distribution for each categorical column
categorical_cols = ["job", "marital", "education", "default", "housing", 
                     "loan", "contact", "month", "day_of_week", "poutcome"]

for col in categorical_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()

--- job ---
job
admin.           10408
blue-collar       9240
technician        6731
services          3963
management        2921
retired           1715
entrepreneur      1453
self-employed     1416
housemaid         1057
unemployed        1009
student            874
Name: count, dtype: int64

--- marital ---
marital
married     24694
single      11494
divorced     4599
Name: count, dtype: int64

--- education ---
education
university.degree      12096
high.school             9464
basic.9y                6006
professional.course     5225
basic.4y                4118
basic.6y                2264
unknown                 1596
illiterate                18
Name: count, dtype: int64

--- default ---
default
no         32348
unknown     8439
Name: count, dtype: int64

--- housing ---
housing
yes        21376
no         18427
unknown      984
Name: count, dtype: int64

--- loan ---
loan
no         33620
yes         6183
unknown      984
Name: count, dtype: int64

--- contact ---
contact
cellu

In [47]:
# ============================================================
# MERGE RARE CATEGORY — education
# ============================================================

In [48]:
# "illiterate" has only 18 observations, too rare to be a stable standalone dummy column
# Merge it into "basic.4y" (the lowest formal education tier) to avoid instability across train/test splits
df["education"] = df["education"].replace("illiterate", "basic.4y")

df["education"].value_counts()

education
university.degree      12096
high.school             9464
basic.9y                6006
professional.course     5225
basic.4y                4136
basic.6y                2264
unknown                 1596
Name: count, dtype: int64

In [49]:
# ============================================================
# ONE-HOT ENCODING — CATEGORICAL FEATURES
# ============================================================

In [50]:
# All categorical columns are nominal (no inherent order), so one-hot encoding is appropriate
# drop_first=True to avoid the dummy variable trap (redundant column that's a linear combination of the others)
categorical_cols = ["job", "marital", "education", "default", "housing", 
                     "loan", "contact", "month", "day_of_week", "poutcome"]

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=float)

print(df.shape)

(40787, 46)


In [51]:
df.head()

,age,campaign,pdays,previous,euribor3m,y,was_previously_contacted,job_blue-collar,job_entrepreneur,job_housemaid,...,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
0,56.0,1,-1,0,4.857,no,0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,57.0,1,-1,0,4.857,no,0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,37.0,1,-1,0,4.857,no,0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,40.0,1,-1,0,4.857,no,0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,56.0,1,-1,0,4.857,no,0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [52]:
# ============================================================
# ENCODE TARGET VARIABLE
# ============================================================

In [53]:
# Convert target to binary: yes -> 1, no -> 0
df["y"] = df["y"].map({"yes": 1, "no": 0})

df["y"].value_counts()

y
0    36193
1     4594
Name: count, dtype: int64

In [54]:
# ============================================================
# TRAIN/TEST SPLIT
# ============================================================

In [55]:
from sklearn.model_selection import train_test_split

In [56]:
X = df.drop(columns=["y"])
y = df["y"] 

In [57]:
# Stratified split to preserve the class imbalance ratio in both sets
X_train,X_test,y_train,y_test = train_test_split(X, y,test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_train.value_counts(normalize=True))

(32629, 45) (8158, 45)
y
0    0.88737
1    0.11263
Name: proportion, dtype: float64
y
0    0.88737
1    0.11263
Name: proportion, dtype: float64


In [58]:
# ============================================================
# FEATURE SCALING (StandardScaler)
# ============================================================

In [59]:
from sklearn.preprocessing import StandardScaler

In [60]:
# Fit only on training data to prevent data leekage from test set statistics

In [61]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, X_test_scaled.shape)
print("Train mean (should be~0):", X_train_scaled.mean().round(3))
print("Train std (should be~1):", X_train_scaled.std().round(3))

(32629, 45) (8158, 45)
Train mean (should be~0): 0.0
Train std (should be~1): 1.0


In [62]:
# ============================================================
# VERIFY SCALING PRECISION
# ============================================================

In [63]:
print("Train mean (raw):", X_train_scaled.mean())
print("Train std (raw):", X_train_scaled.std())

Train mean (raw): 1.2400460125009836e-17
Train std (raw): 1.0


In [64]:
# ============================================================
# VERIFY SCALING PER-COLUMN (more rigorous check)
# ============================================================

In [65]:
# Check mean and std for each column individually, not the flattened overall average
check_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
print(check_df.mean().round(3).describe())  # summary of all column means
print(check_df.std().round(3).describe())   # summary of all column stds

count    45.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
dtype: float64
count    45.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
dtype: float64


In [66]:
# ============================================================
# HANDLE CLASS IMBALANCE (SMOTE) — TRAIN SET ONLY
# ============================================================

In [67]:
from imblearn.over_sampling import SMOTE

In [68]:
# Apply SMOTE only on training data to avoid leaking synthetic patterns into test set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_resampled.value_counts().to_dict())

Before SMOTE: {0: 28954, 1: 3675}
After SMOTE: {0: 28954, 1: 28954}


In [69]:
# ============================================================
# BASELINE SVM MODEL
# ============================================================

In [70]:
from sklearn.svm import SVC
from sklearn.metrics import  classification_report, roc_auc_score

In [71]:
# Baseline model with default hyperparameters (RBF kernel is sklearn's default)
svm_baseline = SVC(random_state=42)
svm_baseline.fit(X_train_resampled, y_train_resampled)

y_pred_baseline = svm_baseline.predict(X_test_scaled)

print(classification_report(y_test,y_pred_baseline))

              precision    recall  f1-score   support

           0       0.94      0.91      0.92      7239
           1       0.41      0.51      0.46       919

    accuracy                           0.86      8158
   macro avg       0.67      0.71      0.69      8158
weighted avg       0.88      0.86      0.87      8158



In [72]:
# ============================================================
# HYPERPARAMETER TUNING — SUBSAMPLE FOR SPEED
# ============================================================

In [73]:
from sklearn.model_selection import GridSearchCV

# SVM's training time grows quadratically/cubically with sample size,
# so we tune hyperparameters on a smaller subsample first, then train
# the final model with the best parameters on the full dataset
np.random.seed(42)
sample_idx = np.random.choice(len(X_train_scaled), size=5000, replace=False)
X_sample = X_train_scaled[sample_idx]
y_sample = y_train.iloc[sample_idx]

print("Sample class distribution:")
print(y_sample.value_counts())

# Apply SMOTE on the sample
X_sample_resampled, y_sample_resampled = smote.fit_resample(X_sample, y_sample)
print("After SMOTE on sample:", y_sample_resampled.value_counts().to_dict())

Sample class distribution:
y
0    4438
1     562
Name: count, dtype: int64
After SMOTE on sample: {0: 4438, 1: 4438}


In [74]:
# ============================================================
# GRIDSEARCHCV — HYPERPARAMETER TUNING ON SAMPLE
# ============================================================

In [75]:
# Parameter grid: C controls margin/error trade-off, gamma controls RBF kernel reach
param_grid = {
    "C": [0.1, 1, 10],
    "gamma": ["scale", 0.01, 0.1],
    "kernel": ["rbf"]
}

grid_search = GridSearchCV(
    SVC(random_state=42),
    param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(X_sample_resampled, y_sample_resampled)

print("Best params:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)

Best params: {'C': 10, 'gamma': 0.1, 'kernel': 'rbf'}
Best CV ROC-AUC: 0.9914922575353188


In [76]:
# ============================================================
# FINAL MODEL — FASTER VERSION (no probability=True)
# ============================================================

In [77]:
# Removed probability=True (Platt scaling adds significant overhead)
# Using decision_function instead for ROC-AUC calculation
# max_iter caps runtime in case convergence is slow
svm_final = SVC(C=10, gamma=0.1, kernel="rbf", max_iter=50000, random_state=42)
svm_final.fit(X_train_resampled, y_train_resampled)

y_pred_final = svm_final.predict(X_test_scaled)

print(classification_report(y_test, y_pred_final))
print("Test ROC-AUC:", roc_auc_score(y_test, svm_final.decision_function(X_test_scaled)))

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.91      0.92      0.91      7239
           1       0.28      0.25      0.27       919

    accuracy                           0.85      8158
   macro avg       0.60      0.58      0.59      8158
weighted avg       0.84      0.85      0.84      8158

Test ROC-AUC: 0.6970747406932074


In [79]:
# Previous search allowed C=10 & gamma=0.1 which overfit badly on the full test set
# Narrowing the range to more conservative values to reduce overfitting risk
param_grid_v2 = {
    "C": [0.1, 1, 5],
    "gamma": ["scale", 0.001, 0.01],
    "kernel": ["rbf"]
}

grid_search_v2 = GridSearchCV(
    SVC(random_state=42),
    param_grid_v2,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search_v2.fit(X_sample_resampled, y_sample_resampled)

print("Best params:", grid_search_v2.best_params_)
print("Best CV ROC-AUC:", grid_search_v2.best_score_)

Best params: {'C': 5, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV ROC-AUC: 0.9729167687286214


In [80]:
# Using more conservative hyperparameters (C=5, gamma='scale') to reduce overfitting risk
# No probability=True to keep training time manageable
svm_final_v2 = SVC(C=5, gamma="scale", kernel="rbf", max_iter=50000, random_state=42)
svm_final_v2.fit(X_train_resampled, y_train_resampled)

y_pred_final_v2 = svm_final_v2.predict(X_test_scaled)

print(classification_report(y_test, y_pred_final_v2))
print("Test ROC-AUC:", roc_auc_score(y_test, svm_final_v2.decision_function(X_test_scaled)))

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.92      0.94      0.93      7239
           1       0.45      0.40      0.42       919

    accuracy                           0.88      8158
   macro avg       0.69      0.67      0.68      8158
weighted avg       0.87      0.88      0.87      8158

Test ROC-AUC: 0.7397907086824616


In [81]:
# ============================================================
# THRESHOLD TUNING
# ============================================================

from sklearn.metrics import precision_recall_curve, f1_score

# Get decision function scores (distance from the hyperplane)
y_scores = svm_final_v2.decision_function(X_test_scaled)

# Compute precision-recall pairs for different thresholds
precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores)

# Find the threshold that maximizes F1 score
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Best threshold: {best_threshold:.4f}")
print(f"Precision at best threshold: {precisions[best_idx]:.3f}")
print(f"Recall at best threshold: {recalls[best_idx]:.3f}")
print(f"F1 at best threshold: {f1_scores[best_idx]:.3f}")

Best threshold: -0.4552
Precision at best threshold: 0.378
Recall at best threshold: 0.535
F1 at best threshold: 0.443


In [82]:
# ============================================================
# CROSS-VALIDATION (ON SUBSAMPLE FOR SPEED)
# ============================================================

In [83]:
from sklearn.model_selection import cross_val_score

# Running CV on a subsample (not the full 58k rows) due to SVM's computational cost
# This gives an approximate measure of model stability across folds
np.random.seed(42)
cv_sample_idx = np.random.choice(len(X_train_scaled), size=12000, replace=False)
X_cv_sample = X_train_scaled[cv_sample_idx]
y_cv_sample = y_train.iloc[cv_sample_idx]

# Apply SMOTE on this subsample
X_cv_resampled, y_cv_resampled = smote.fit_resample(X_cv_sample, y_cv_sample)

svm_cv = SVC(C=5, gamma="scale", kernel="rbf", max_iter=50000, random_state=42)
cv_scores = cross_val_score(svm_cv, X_cv_resampled, y_cv_resampled, cv=5, scoring="roc_auc", n_jobs=-1)

print("CV ROC-AUC scores:", cv_scores)
print(f"CV Mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

CV ROC-AUC scores: [0.92954361 0.97588462 0.981286   0.98215257 0.97902908]
CV Mean: 0.9696 ± 0.0201


In [84]:
# ============================================================
# TEST SET ROC-AUC (for direct comparison)
# ============================================================

In [85]:
test_roc_auc = roc_auc_score(y_test, svm_final_v2.decision_function(X_test_scaled))
print(f"Test ROC-AUC: {test_roc_auc:.4f}")

Test ROC-AUC: 0.7398
